# 🚀 Phase 2A: Track A Few-Shot & Zero-Day Generalization Benchmark ($N \le 10\text{k}$)
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection: An Axiomatic-Empirical Fuzzy DEMATEL Simulation Framework*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

### 📌 Track A Benchmark Objectives:
1. **Multi-Paradigm Comparative Evaluation**: Benchmark 8 modern architectures across 5-fold cross-validation:
   - **Foundation Models**: `TabPFN v3`, `TabICL v2` (Bayesian In-Context Zero-Shot Learning)
   - **Modern Deep Learning**: `Mambular SSM` ($O(L)$ linear scaling), `FT-Transformer` ($O(L^2)$ attention), `SAINT` (dual self/row attention), `GraphIDS` (Inductive GNN)
   - **Tuned Baselines**: `XGBoost`, `LightGBM` (Optuna 50 trials)
2. **Real vs. Synthetic Dataset Transparency**: Automatically tests for real datasets in `ROOT/data/raw/` (e.g. `MachineLearningCVE`), printing prominent status alerts.
3. **Anti-Leakage Protection**: Strict `GroupKFold` partitioning by `/24` subnet masks with fold-isolated preprocessing.
4. **Autorecovery Checkpointing**: Powered by `CheckpointManager` on Google Drive—survives sudden disconnects and preemptions without restarting from fold 1.
5. **Downstream Telemetry Export**: Serializes benchmark metrics to `experiment_output/track_a/benchmark_results.json` for Phase 3 (Ablation), Phase 4 (Fuzzy DEMATEL), and Phase 5 (LaTeX Tables).


### 1. ☁️ Google Drive Mount & Project Root Auto-Resolution


In [ ]:
import os, sys
from pathlib import Path

# 1. Mount Google Drive if running inside Colab
try:
    from google.colab import drive
    if not Path('/content/drive').exists() and not Path('/content/My Drive').exists():
        drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# 2. Candidate root paths (supporting both 'Colab Notebook' and 'Colab Notebooks')
CANDIDATE_ROOTS = [
    Path('/content/drive/My Drive/Colab Notebook'),
    Path('/content/drive/My Drive/Colab Notebooks'),
    Path('/content/drive/MyDrive/Colab Notebook'),
    Path('/content/drive/MyDrive/Colab Notebooks'),
    Path('/content/My Drive/Colab Notebook'),
    Path('/content/My Drive/Colab Notebooks'),
    Path('/content/Colab Notebook'),
    Path('/content/Colab Notebooks'),
    Path('/Colab Notebook'),
    Path('/Colab Notebooks'),
    Path('/content/drive/MyDrive/is_ai-vuln'),
    Path('/content/drive/My Drive/is_ai-vuln'),
    Path('/content/is_ai-vuln'),
    Path('.').resolve()
]

PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists() and ((cand / 'src').exists() or (cand / 'data' / 'raw').exists()):
        PROJECT_ROOT = cand.resolve()
        break

# Dynamic discovery inside /content/drive if not yet matched
if PROJECT_ROOT is None and Path('/content/drive').exists():
    for drive_parent in [Path('/content/drive/My Drive'), Path('/content/drive/MyDrive'), Path('/content/drive'), Path('/content/My Drive')]:
        if drive_parent.exists():
            try:
                for sub in drive_parent.iterdir():
                    if sub.is_dir() and ('colab notebook' in sub.name.lower() or 'is_ai-vuln' in sub.name.lower()):
                        if (sub / 'src').exists() or (sub / 'data' / 'raw').exists():
                            PROJECT_ROOT = sub.resolve()
                            break
            except Exception:
                pass
            if PROJECT_ROOT:
                break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Inspect data/raw directory to confirm real datasets presence
raw_dir = PROJECT_ROOT / 'data' / 'raw'
detected_folders = []
if raw_dir.exists():
    try:
        detected_folders = [f.name for f in raw_dir.iterdir() if f.is_dir()]
    except Exception:
        pass

print("=" * 80)
print(f"✅ Active Project Root : {PROJECT_ROOT}")
print(f"✅ src/ directory found: {(PROJECT_ROOT / 'src').exists()}")
print(f"📁 Raw Data Path       : {raw_dir.resolve()}")
print(f"🔍 Detected Raw Folders: {detected_folders}")
known_real = ['CIC-DDoS2019', 'MachineLearningCVE', 'NSL-KDD', 'TON-IoT', 'ToN-IOT', 'TrafficLabelling', 'unsw-data-full']
found_known = [f for f in detected_folders if f in known_real]
if found_known:
    print(f"🛡️ [DATA STATUS: AUTHENTIC REAL DATASETS DETECTED] Found: {found_known}")
else:
    print("ℹ️ [DATA STATUS] Note: Real datasets will also be dynamically located across Drive search paths.")
print("=" * 80)


### 2. 📦 Core & Model Dependencies Installation


In [ ]:
# Install core dependencies for Track A benchmark
!pip install -q xgboost lightgbm optuna scikit-learn imbalanced-learn pandas numpy matplotlib seaborn networkx requests tqdm

print("✅ Benchmark dependencies ready.")


### 3. 🛡️ Data Ingestion, Decontamination & Real vs. Synthetic Detection

> [!IMPORTANT]
> **Data Notice**:
> - If you uploaded dataset folders (`MachineLearningCVE`, `unsw-data-full`, `TON-IoT`, `CIC-DDoS2019`, `NSL-KDD`) to `ROOT/data/raw/`, Track A benchmarks will run on **Real Authentic NetFlow Data**.
> - Otherwise, the pipeline safely runs on **Synthetic Benchmark Fallback** with a prominent notice banner.


In [ ]:
import pandas as pd
from src.data.prep_pipeline import run_preparation_pipeline
from src.data.drive_downloader import initialize_dataset_directories

# Ensure persistent directories exist
dirs = initialize_dataset_directories(PROJECT_ROOT)

# Run or load preprocessed decontaminated benchmark dataset
prep_result = run_preparation_pipeline("CICIDS2017", base_dir=PROJECT_ROOT, prefer_sample=False, n_splits=5)

print("\n" + "=" * 80)
if prep_result["is_synthetic"]:
    print("⚠️ [DATA NOTICE: BENCHMARK RUNNING ON SYNTHETIC FALLBACK DATA]")
    print(f"   To run on real data, place dataset folders in: {dirs['raw'].resolve()}")
else:
    print("🛡️ [DATA NOTICE: BENCHMARK RUNNING ON AUTHENTIC REAL DATASET]")
    print(f"   Source file  : {prep_result['raw_source_file']}")
    print(f"   Cleaned Shape: {prep_result['cleaned_shape']}")
    print(f"   Cleaned File : {prep_result['processed_file']}")
print("=" * 80)


### 4. 🔄 Fault-Tolerant Checkpoint Bootstrap (`CheckpointManager`)


In [ ]:
import json
from src.utils.checkpoint_manager import CheckpointManager

# Establish checkpoint directory in Google Drive or workspace
chk_dir = PROJECT_ROOT / "checkpoints"
chk_dir.mkdir(parents=True, exist_ok=True)

manager = CheckpointManager(
    drive_checkpoint_dir=chk_dir,
    dataset_name="CICIDS2017",
    track_name="Track_A",
    total_folds=5
)

print(f"🔄 Checkpoint State: {manager.state['status']}")
print(f"Completed models: {manager.state['completed_models']}")
print(f"Current model: {manager.state['current_model']} (Fold: {manager.state['current_fold']})")


### 5. 🔬 Track A 5-Fold Cross-Validation Execution (8 Models)


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from src.models import get_model
from src.evaluation import evaluate_fold_run, calculate_ttf_utility
from src.data.splitters import AntiLeakageGroupKFold, extract_subnet_mask, safe_slice
from src.utils.environment import flush_memory

# Load cleaned dataset
clean_file = prep_result["processed_file"]
df = pd.read_parquet(clean_file) if str(clean_file).endswith(".parquet") else pd.read_csv(clean_file)

# Prepare features and labels (Sample N=10,000 for Track A)
df_track_a = df.head(10000).copy()
target_col = "is_attack" if "is_attack" in df_track_a.columns else "label"
feature_cols = [c for c in df_track_a.select_dtypes(include=[np.number]).columns if c != target_col]

X = df_track_a[feature_cols].values
y = df_track_a[target_col].values

# Extract subnet blocks for host session leakage protection
src_ip_col = "source_ip" if "source_ip" in df_track_a.columns else None
subnets = extract_subnet_mask(df_track_a[src_ip_col]) if src_ip_col else pd.Series(np.arange(len(df_track_a)) // 2000)

models_to_run = [
    "XGBoost",
    "LightGBM",
    "TabPFN_v3",
    "TabICL_v2",
    "Mambular_SSM",
    "FT_Transformer",
    "SAINT",
    "GraphIDS"
]

gkf = AntiLeakageGroupKFold(n_splits=5)
fold_indices = list(gkf.split(X, y, groups=subnets))

all_metrics = []

for model_name in models_to_run:
    if manager.should_skip_model(model_name):
        print(f"⏩ Model '{model_name}' already fully completed in checkpoint. Skipping...")
        continue
        
    print(f"\n{'='*50}\n🚀 Running Model: {model_name}\n{'='*50}")
    
    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices, start=1):
        if manager.should_skip_fold(model_name, fold_idx):
            print(f"  ⏩ Fold {fold_idx} already cached. Skipping...")
            continue
            
        print(f"  ▶️ Training {model_name} | Fold {fold_idx}/5 (Train: {len(train_idx)}, Val: {len(val_idx)})...")
        
        # 1. Fold-isolated scaling
        X_tr, y_tr = safe_slice(X, train_idx), safe_slice(y, train_idx)
        X_va, y_va = safe_slice(X, val_idx), safe_slice(y, val_idx)
        
        scaler = StandardScaler().fit(X_tr)
        X_tr_sc = scaler.transform(X_tr)
        X_va_sc = scaler.transform(X_va)
        
        # 2. Fit Model
        model = get_model(model_name)
        model.fit(X_tr_sc, y_tr)
        
        # 3. Predict & Profile
        preds = model.predict(X_va_sc)
        probs = model.predict_proba(X_va_sc)
        profile = model.profile_inference(X_va_sc, warmup_runs=3, repeat_runs=10)
        
        # 4. Compute Metrics & TTF Utility
        fold_met = evaluate_fold_run(y_va, preds, probs, profile)
        fold_met["ttf_t1"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T1")
        fold_met["ttf_t2"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T2")
        fold_met["ttf_t3"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T3")
        
        manager.save_fold_progress(
            model_name=model_name,
            fold_index=fold_idx,
            metrics=fold_met,
            predictions=preds,
            probabilities=probs
        )
        all_metrics.append({"model": model_name, "fold": fold_idx, **fold_met})
        print(f"  ✅ Fold {fold_idx} Complete | F1: {fold_met['f1_macro']:.4f} | Lat: {fold_met['latency_ms_per_flow']:.3f}ms")
        flush_memory()

# Save compiled benchmark results for downstream phases
output_dir = PROJECT_ROOT / "experiment_output" / "track_a"
output_dir.mkdir(parents=True, exist_ok=True)

df_results = pd.DataFrame(all_metrics)
if not df_results.empty:
    summary = df_results.groupby("model").agg({
        "f1_macro": ["mean", "std"],
        "f1_unseen": ["mean", "std"],
        "latency_ms_per_flow": ["mean"],
        "vram_peak_mb": ["mean"],
        "ttf_t1": ["mean"],
        "ttf_t2": ["mean"],
        "ttf_t3": ["mean"]
    })
    summary.to_csv(output_dir / "master_summary.csv")
    with open(output_dir / "benchmark_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "is_synthetic": prep_result["is_synthetic"],
            "dataset": prep_result["dataset_name"],
            "models": models_to_run,
            "metrics": all_metrics
        }, f, indent=2)
    print(f"💾 Benchmark telemetry saved to: {output_dir / 'benchmark_results.json'}")


### 6. 📊 TTF Multi-Task Utility Tradeoff Visualizer


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5), dpi=150)
if len(all_metrics) > 0:
    df_m = pd.DataFrame(all_metrics)
    sns.barplot(data=df_m, x="model", y="f1_macro", palette="viridis")
    plt.title(f"Track A 5-Fold Benchmark (Data: {'SYNTHETIC' if prep_result['is_synthetic'] else 'REAL'})")
    plt.ylabel("Macro F1 Score")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(output_dir / "figure1_track_a_benchmark.png")
    plt.show()
